# Modellierung & Evaluation

Verglichen werden klassische Pipelines und generative DeepTabular-Modelle, um die Hypothesen zu Kredit-Scoring-Daten zu testen:
- **H1:** Klassische ML-Modelle erzielen vergleichbare oder bessere Ergebnisse als generative Modelle auf Tabulardaten.
- **H2:** Generative Modelle verursachen einen deutlich höheren Ressourcenbedarf gemessen an Trainingszeit/Rechenaufwand.

## 1️⃣ Vorgehensweise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from deeptabular.models import MambularClassifier, FTTransformerClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)
import time

In [ ]:
train_df = pd.read_csv("../data/train/train_fe.csv")
test_df = pd.read_csv("../data/test/test_fe.csv")

print(f"Trainigsdatensatz: {train_df.shape}")
print(f"Testdatensatz: {test_df.shape}")

## 2️⃣ Train-Test Split

In [ ]:
X = train_df.drop("Credit_Score", axis=1)
y = train_df["Credit_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

print(f"X_Trainigsdatensatz: {X_train.shape}")
print(f"X_Testdatensatz: {X_test.shape}")
print(f"y_Trainigsdatensatz: {y_train.shape}")
print(f"y_Testdatensatz: {X_test.shape}")

## 3️⃣ Pipelines

### Preprocessing: Numerische & Kategorische Pipelines

- Numerische Features enthalten alle kontinuierlichen und zählenden Engineering-Kennzahlen (z. B. Einkommens-, Delay- und Ratio-Variablen).
- Kategorisch bleiben `Month`, `Occupation` und `Credit_Mix`.
- Verarbeitung: Numerische Werte werden median-imputet und skaliert, kategorische Werte werden mit dem häufigsten Wert imputet und anschließend One-Hot-encodiert.


In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print(f"{len(num_cols)} numerische Merkmale und {len(cat_cols)} kategorielle Merkmale identifiziert.")
print(f"Kategorisch: {cat_cols}")
print(f"Numerisch: {num_cols}")

In [ ]:
numeric_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

In [ ]:
categorical_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    remainder="drop",
)

preprocessor

### Modell-Pipelines

- **LR**: Logistic Regression als lineares Baseline-Modell.
- **RF**: RandomForestClassifier als robuster, baumbasierter Klassifikator.
- **HGB**: HistGradientBoostingClassifier als leistungsfähiger Gradient-Boosting-Ansatz.
- **MAM**: MambularClassifier aus DeepTabular für sequentielle Mamba-Blöcke.
- **FTT**: FTTransformerClassifier als attention-basiertes DeepTabular-Modell.


In [ ]:
pipe_lr = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(
        random_state=42,
    )),
])

pipe_lr

In [ ]:
pipe_rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
    )),
])

pipe_rf

In [ ]:
pipe_hgb = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])

pipe_hgb

Deeptabular-Modelle übernehmen eigenständig die Proprocessing-Schritte

In [ ]:
pipe_mam = Pipeline(steps=[
    ("model", MambularClassifier(
        numerical_preprocessing="standardization",
        categorical_preprocessing="one-hot",
    )),
])

In [ ]:
pipe_ftt = Pipeline(steps=[
    ("model", FTTransformerClassifier(
        numerical_preprocessing="standardization",
        categorical_preprocessing="one-hot",
    )),
])

## 4️⃣ Hyperparameter-Tuning

In [ ]:
param_grid_lr = {
    "model__C": [0.1, 1.0],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [200, 500],
}

param_grid_rf = {
    "model__n_estimators": [200],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 4],
    "model__class_weight": [None, "balanced"],
}

param_grid_hgb = {
    "model__learning_rate": [0.03, 0.1],
    "model__max_iter": [100],
    "model__max_leaf_nodes": [31, 127],
    "model__l2_regularization": [0.0, 1.0],
}

param_grid_mam = {
    "model__d_model": [64, 128],
    "model__n_layers": [2, 4],
    "model__lr": [1e-3, 1e-4],
}

param_grid_ftt = {
    "model__d_model": [64, 128],
    "model__n_layers": [2, 4],
    "model__lr": [1e-3, 1e-4],
}

In [ ]:
def run_grid_clf_with_time(name, pipe, grid):
    """
    GridSearchCV für Klassifikation, inkl. Zeitmessung.
    Funktioniert für binary UND multiclass.
    """
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        verbose=0,
    )
    
    # Train
    t0 = time.perf_counter()
    gs.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    # Predict
    t1 = time.perf_counter()
    y_pred = gs.predict(X_test)
    pred_time = time.perf_counter() - t1

    # Anzahl Klassen prüfen
    n_classes = len(np.unique(y_test))
    avg = "binary" if n_classes == 2 else "weighted"

    # Metriken
    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average=avg)

    # ROC-AUC
    try:
        y_proba = gs.predict_proba(X_test)
        if n_classes == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            # One-vs-rest, gewichteter Durchschnitt
            auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        auc = np.nan

    print(f"\n{name}")
    print("-" * 60)
    print(f"Best params:          {gs.best_params_}")
    print(f"CV best bal-acc:      {gs.best_score_:.4f}")
    print(f"Test Accuracy:        {acc:.4f}")
    print(f"Test Balanced Acc.:   {bacc:.4f}")
    print(f"Test F1 ({avg}):      {f1:.4f}")
    if not np.isnan(auc):
        print(f"Test ROC-AUC:         {auc:.4f}")
    else:
        print(f"Test ROC-AUC:         n/a")
    print(f"Train time (s):       {train_time:.2f}")
    print(f"Predict time (s):     {pred_time:.4f}")

    return {
        "name": name,
        "grid": gs,
        "acc": acc,
        "bacc": bacc,
        "f1": f1,
        "auc": auc,
        "train_time": train_time,
        "pred_time": pred_time,
        "cv_best_bal_acc": gs.best_score_,
    }

In [ ]:
results = []

results.append(run_grid_clf_with_time("LR",  pipe_lr,  param_grid_lr))
results.append(run_grid_clf_with_time("RF",  pipe_rf,  param_grid_rf))
results.append(run_grid_clf_with_time("HGB", pipe_hgb, param_grid_hgb))

In [ ]:
results.append(run_grid_clf_with_time("MAM", pipe_mam, param_grid_mam))
results.append(run_grid_clf_with_time("FTT", pipe_ftt, param_grid_ftt))

In [ ]:
df_results = pd.DataFrame(results)

family_map = {
    "LR":  "klassisch",
    "RF":  "klassisch",
    "HGB": "klassisch",
    "MAM": "generativ",
    "FTT": "generativ",
}

df_results["family"] = df_results["name"].map(family_map)

print("\nEinzelne Modelle:")
display(df_results[["name", "family", "bacc", "f1", "auc", "train_time", "pred_time"]])

print("\nGruppensicht nach Modellfamilie:")
df_family = (
    df_results
    .groupby("family")[["bacc", "f1", "auc", "train_time", "pred_time"]]
    .mean()
    .sort_values("bacc", ascending=False)
)
display(df_family)